# Email RAG Training Pipeline — Template

Train a model to search and answer questions over your email corpus using reinforcement learning.

This notebook walks through the full pipeline:

1. **Upload corpus** — Point to your email data (.mbox or folder of emails)
2. **Generate synthetic QA** — Create question-answer training pairs from your corpus
3. **Define your environment** — Build a custom `SearchEnv` with citation rewards
4. **Launch training** — Submit the RL fine-tuning job to CGFT

This is a **template** — replace the placeholder values with your own data and customize the reward function to match your use case.

## Setup

Install the CGFT client library and configure credentials.

In [ ]:
# Google Colab — install the cgft package directly
!pip install git+https://github.com/cgftinc/cgft.git

In [ ]:
# Local development only (skip this cell on Colab)
# %load_ext autoreload
# %autoreload 2
#
# import sys
# from pathlib import Path
#
# cwd = Path.cwd()
# candidate_roots = [cwd, cwd.parent]
# repo_root = next((p for p in candidate_roots if (p / "src" / "cgft").exists()), cwd)
# src_path = repo_root / "src"
# if src_path.exists() and str(src_path) not in sys.path:
#     sys.path.insert(0, str(src_path))

In [ ]:
from cgft.utils import ensure_safe_python_version

ensure_safe_python_version()

In [ ]:
# ── Configuration ────────────────────────────────────────────────────
# Create an API key at https://app.cgft.io/account/api-keys

API_KEY = "sk_YOUR_CGFT_API_KEY"
BASE_URL = "https://app.cgft.io"

# LLM endpoint for synthetic QA generation (OpenAI-compatible)
LLM_API_KEY = "YOUR_LLM_API_KEY"
LLM_BASE_URL = "https://api.openai.com/v1"  # or your Azure/other endpoint
LLM_MODEL = "gpt-4o"

# LLM endpoint for the judge model (used during training for reward scoring).
# Can be the same as above, or a faster/cheaper model.
JUDGE_API_KEY = LLM_API_KEY
JUDGE_BASE_URL = LLM_BASE_URL
JUDGE_MODEL = "gpt-4o-mini"

# ── Corpus ────────────────────────────────────────────────────────────
# Path to your email data. Supported formats:
#   - A folder of .mbox files  (Gmail Takeout, Thunderbird export, etc.)
#   - A folder of .eml files
#   - A .jsonl file with one email per line (keys: subject, body, from, to, date, thread_id)
EMAIL_DATA_PATH = "./path/to/your/emails"  # <── REPLACE
CORPUS_NAME = "my-email-corpus"             # <── REPLACE

# Brief description of the corpus (used by the QA generator to write better questions)
CORPUS_DESCRIPTION = "Work emails from the engineering team at Acme Corp."
EXAMPLE_QUERIES = [
    "when did Alice last email me about the deployment?",
    "what was agreed in the Q3 planning thread?",
    "what were the action items from the security review?",
]

# ── QA Generation ────────────────────────────────────────────────────
TOTAL_SAMPLES = 200          # total synthetic QA pairs to generate
OUTPUT_DIR = "outputs/email_rag"

# ── Training ──────────────────────────────────────────────────────────
EXPERIMENT_NAME = "email-rag-v1"

---
## Step 1: Upload Corpus

Parse and chunk your emails, then upload to the CGFT Corpus API for BM25 search.

Pick **one** of the options below depending on your data format.

### Option A: .mbox files

If your emails are exported as `.mbox` (e.g. Gmail Takeout), use the built-in
preprocessing pipeline: convert → dedupe → filter automated emails → clean bodies → chunk.

In [ ]:
from pathlib import Path

from cgft.chunkers.email import EmailChunker
from cgft.chunkers.models import ChunkCollection
from cgft.preprocess.email import mbox
from cgft.preprocess.email.dedupe import dedupe_email_folder
from cgft.preprocess.email.clean_bodies import clean_email_folder

# 1. Convert .mbox → .jsonl (one email per line)
mbox.convert_mbox_folder(EMAIL_DATA_PATH)

# 2. Deduplicate across all .jsonl files
dedupe_email_folder(EMAIL_DATA_PATH)

# 3. Clean body text (strip quoted replies, signatures)
clean_email_folder(EMAIL_DATA_PATH, input_name="_deduped.jsonl", output_name="_cleaned.jsonl")

# 4. Chunk into thread-aware segments
email_chunker = EmailChunker()
chunks = email_chunker.chunk_file(Path(EMAIL_DATA_PATH) / "_cleaned.jsonl")
collection = ChunkCollection(chunks)
print(f"Chunked → {len(chunks)} chunks")

### Option B: Pre-processed .jsonl or folder of documents

If your emails are already in a folder of text/markdown files or a JSONL, you can
chunk them directly with the markdown chunker, or bring your own `ChunkCollection`.

In [ ]:
from cgft.chunkers.markdown import MarkdownChunker
from cgft.chunkers.models import ChunkCollection

# Chunk a folder of text/markdown files
chunker = MarkdownChunker()
chunks = chunker.chunk_folder(EMAIL_DATA_PATH)
collection = ChunkCollection(chunks)
print(f"Chunked → {len(chunks)} chunks")

### Upload chunks to CGFT Corpus API

In [ ]:
from cgft.corpus.corpora.source import CorporaChunkSource

source = CorporaChunkSource(api_key=API_KEY, corpus_name=CORPUS_NAME, base_url=BASE_URL)
source.populate_from_chunks(collection)
print(f"Corpus ID: {source.corpus_id}")

---
## Step 2: Generate Synthetic QA

The `CgftPipeline` reads chunks from your corpus, profiles the content, and generates
diverse question-answer pairs with built-in quality filtering.

In [ ]:
from cgft.qa_generation.cgft_models import (
    CgftPipelineConfig,
    CorpusConfig,
    CorpusContextConfig,
    EntityExtractionLLMConfig,
    FilteringConfig,
    GenerationConfig,
    GroundingLLMFilterConfig,
    LinkerConfig,
    LLMDirectGenerationConfig,
    LLMEnvGenerationConfig,
    LLMGuidedLinkerConfig,
    OutputConfig,
    PlatformConfig,
    RefinementConfig,
    RetrievalLLMFilterConfig,
    TargetsConfig,
    TransformationConfig,
)
from cgft.qa_generation.cgft_pipeline import CgftPipeline

MAX_CONCURRENT = 8  # parallel LLM calls per pipeline stage

cfg = CgftPipelineConfig(
    platform=PlatformConfig(
        api_key=API_KEY,
        base_url=BASE_URL,
        llm_api_key=LLM_API_KEY,
        llm_base_url=LLM_BASE_URL,
    ),
    corpus=CorpusConfig(corpus_name=CORPUS_NAME, min_chunk_chars=400),
    corpus_context=CorpusContextConfig(
        description=CORPUS_DESCRIPTION,
        example_queries=EXAMPLE_QUERIES,
        model=LLM_MODEL,
        entity_extraction_llm=EntityExtractionLLMConfig(model=LLM_MODEL),
    ),
    linker=LinkerConfig(llm_guided=LLMGuidedLinkerConfig(model=LLM_MODEL)),
    generation=GenerationConfig(
        llm_direct=LLMDirectGenerationConfig(
            model=LLM_MODEL,
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
            show_batch_progress=True,
        ),
        llm_env=LLMEnvGenerationConfig(model=LLM_MODEL),
    ),
    transformation=TransformationConfig(model=LLM_MODEL, validation_model=LLM_MODEL),
    filtering=FilteringConfig(
        retrieval_llm=RetrievalLLMFilterConfig(
            judge_model=LLM_MODEL,
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
        ),
        grounding_llm=GroundingLLMFilterConfig(
            judge_model=LLM_MODEL,
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
        ),
    ),
    refinement=RefinementConfig(model=LLM_MODEL),
    targets=TargetsConfig(total_samples=TOTAL_SAMPLES),
    output=OutputConfig(dir=OUTPUT_DIR, train_jsonl="train.jsonl", eval_jsonl="eval.jsonl"),
)

cfg.resolve_api_keys()
print(f"Model: {cfg.generation.llm_direct.model}")
print(f"Target samples: {TOTAL_SAMPLES}")
print(f"Filter chain: {cfg.filtering.filters}")

In [ ]:
cgft = CgftPipeline(cfg, source_factory=lambda _cfg: source)
result = cgft.run()

train_data = result["train_dataset"]
eval_data = result["eval_dataset"]
stats = result["stats"]

print(f"Generated: {len(train_data)} train / {len(eval_data)} eval")
print(f"Passed: {stats.get('passed_total')} | Rejected: {stats.get('rejected_total')}")

In [ ]:
# Quick spot check
for qa in train_data[:5]:
    print(f"Q: {qa['question']}")
    print(f"A: {qa['answer'][:200]}...")
    print()

---
## Step 3: Define Your Search Environment

The environment defines:
- **System prompt** — instructions the model sees at inference time
- **Tools** — what the model can call (here: a `search` tool over your corpus)
- **Reward function** — how the model's output is scored during RL training

Below we extend `SearchEnv` with a multi-component reward that scores:

| Component | What it measures |
|---|---|
| **answer_correctness** | LLM judge: is the answer factually correct vs. ground truth? |
| **conciseness** | LLM judge: is the answer direct and not overly verbose? |
| **citation_recall** | Did the model cite the correct source threads? |
| **citation_precision** | Are all cited sources actually relevant? |
| **search_efficiency** | Did the model find the answer without excessive searches? |

Customize the weights, system prompt, and reward logic for your use case.

In [ ]:
import asyncio
import re
from typing import Any

import aiohttp
from benchmax.envs.types import ToolDefinition
from openai import AsyncOpenAI

from cgft.envs.search_env import SearchEnv


# ── System prompt the model sees during training ────────────────────
SYSTEM_PROMPT = """Answer the given question by searching over an email corpus.

You will first reason about the question inside <think> and </think> tags.
Then call the search tool with <tool_call> to retrieve relevant emails.
After reviewing results, either search again or provide your final answer
inside <answer> and </answer> tags.

Cite supporting email threads as [Source: <thread_id>].
You can search up to 4 times."""


# ── Reward weights (tune these for your use case) ──────────────────
W_CORRECTNESS = 1.0       # most important: is the answer right?
W_CONCISENESS = 0.5       # penalize verbose answers
W_CITATION_RECALL = 0.5   # did the model cite the right sources?
W_CITATION_PRECISION = 0.5  # are cited sources actually relevant?
W_SEARCH_EFFICIENCY = 0.1 # bonus for fewer search calls


# ── Helper functions ───────────────────────────────────────────────
_ANSWER_TAG_RE = re.compile(r"<answer>(.*?)</answer>", re.DOTALL | re.IGNORECASE)
_CITATION_RE = re.compile(r"\[(?:Source|Thread)\s*:\s*([^\]]+)\]", re.IGNORECASE)


def _extract_answer_block(text: str) -> str:
    """Pull text from <answer>...</answer> tags, or return full text if no tags."""
    match = _ANSWER_TAG_RE.search(text or "")
    return (match.group(1) if match else text).strip()


def _extract_completion_text(completion: str | list[dict]) -> str:
    """Get the model's text output from either a string or message list."""
    if isinstance(completion, str):
        return completion
    return "\n".join(
        msg.get("content", "")
        for msg in completion
        if isinstance(msg, dict) and msg.get("role") == "assistant"
    )


def _count_search_calls(completion: str | list[dict]) -> int:
    """Count how many times the model called the search tool."""
    if not isinstance(completion, list):
        return 0
    return sum(
        msg.get("content", "").count("<tool_call>")
        for msg in completion
        if isinstance(msg, dict) and msg.get("role") == "assistant"
    )


def _parse_citations(text: str) -> set[str]:
    """Extract cited source IDs from [Source: ...] tags."""
    return {m.group(1).strip() for m in _CITATION_RE.finditer(text or "")}


def _extract_reference_thread_ids(reference_chunks: list) -> set[str]:
    """Get ground-truth thread IDs from the dataset's reference_chunks."""
    ids = set()
    for chunk in reference_chunks:
        if not isinstance(chunk, dict):
            continue
        thread_id = str(chunk.get("metadata", {}).get("thread_id", "")).strip()
        if thread_id:
            ids.add(thread_id)
    return ids


async def _judge_rubric(
    question: str,
    ground_truth: str,
    response: str,
    rubric_title: str,
    rubric_description: str,
    judge_model: str,
    judge_base_url: str,
    judge_api_key: str,
    timeout: float = 30.0,
) -> float:
    """Ask an LLM judge to score a response against a rubric. Returns 0.0 or 1.0."""
    prompt = (
        f"You are evaluating a response against a quality criterion.\n\n"
        f"**Criterion**: {rubric_title}\n"
        f"**Description**: {rubric_description}\n\n"
        f"**Question**: {question}\n"
        f"**Ground Truth**: {ground_truth}\n\n"
        f"**Response**: {response}\n\n"
        f'Score 1 if the response meets the criterion, 0 otherwise.\n'
        f'Respond with JSON: {{"score": 0 or 1, "reasoning": "..."}}'
    )
    client = AsyncOpenAI(base_url=judge_base_url, api_key=judge_api_key)
    try:
        resp = await client.chat.completions.create(
            model=judge_model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            timeout=timeout,
        )
        import json
        content = resp.choices[0].message.content or ""
        # Extract JSON from response
        match = re.search(r'\{.*\}', content, re.DOTALL)
        if match:
            result = json.loads(match.group(0))
            return max(0.0, min(1.0, float(result.get("score", 0))))
    except Exception:
        pass
    return 0.0


# ── The Environment ────────────────────────────────────────────────

class MyEmailSearchEnv(SearchEnv):
    """Email search env with citation + judge-based rewards.

    Extends SearchEnv to add:
    - A BM25 search tool that queries the CGFT Corpus API
    - A multi-component reward function with LLM judge scoring
    """

    system_prompt: str = SYSTEM_PROMPT

    def __init__(
        self,
        *,
        api_key: str,
        corpus_id: str,
        base_url: str,
        judge_base_url: str = "",
        judge_api_key: str = "",
        judge_model: str = "",
        **kwargs,
    ):
        self._api_key = api_key
        self._corpus_id = corpus_id
        self._base_url = base_url.rstrip("/")
        self._judge_base_url = judge_base_url
        self._judge_api_key = judge_api_key
        self._judge_model = judge_model

        # Define the search tool the model can call
        search_tool = ToolDefinition(
            name="search",
            description="Search the email corpus using BM25.",
            input_schema={
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query string.",
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Max results to return (default 10).",
                    },
                },
                "required": ["query"],
                "additionalProperties": False,
            },
        )
        self._tools = {"search": (search_tool, self._search_tool)}

    async def _search_tool(self, query: str, limit: int = 10, **kwargs) -> str:
        """Call the CGFT Corpus API to search your indexed emails."""
        if not query:
            return "Error: Missing required parameter: 'query'"

        url = f"{self._base_url}/api/corpora/{self._corpus_id}/search"
        headers = {
            "Authorization": f"Bearer {self._api_key}",
            "Content-Type": "application/json",
        }

        try:
            async with aiohttp.ClientSession() as session:
                async with session.post(
                    url,
                    json={"query": query, "limit": limit},
                    headers=headers,
                    timeout=aiohttp.ClientTimeout(total=10.0),
                ) as resp:
                    if resp.status != 200:
                        error_text = await resp.text()
                        return f"Error: API returned {resp.status}: {error_text}"
                    data = await resp.json()

            results = data.get("results", [])
            if not results:
                return "No results found."

            lines = []
            for i, item in enumerate(results, 1):
                score = item.get("score")
                score_str = f"(score: {score:.2f})" if score is not None else ""
                lines.append(f"{i}. {item.get('filename', '—')} {score_str}")
                lines.append(f"   Content: {item.get('content', '')}")
                if item.get("metadata"):
                    lines.append(f"   Metadata: {item['metadata']}")
            lines.append(f"\nTotal: {data.get('total', 0)} results")
            return self._truncate_tool_output("\n".join(lines))

        except Exception as e:
            return f"Error: {e}"

    async def compute_reward(
        self,
        rollout_id: str,
        completion: str | list[dict[str, Any]],
        ground_truth: Any,
        **kwargs: Any,
    ) -> dict[str, float]:
        """Score the model's output using judge + citation metrics.

        Returns a dict of reward components. The trainer sums these to get
        the total reward signal — so each weight controls how much that
        component matters relative to the others.
        """
        zero = {
            "answer_correctness": 0.0,
            "conciseness": 0.0,
            "citation_recall": 0.0,
            "citation_precision": 0.0,
            "search_efficiency": 0.0,
        }

        try:
            completion_text = _extract_completion_text(completion)
            if not completion_text.strip():
                return zero

            answer_block = _extract_answer_block(completion_text)

            # ── Citation precision & recall ──────────────────────────
            cited_ids = _parse_citations(answer_block)
            ref_ids = _extract_reference_thread_ids(
                kwargs.get("reference_chunks", [])
            )
            overlap = cited_ids & ref_ids
            citation_recall = (len(overlap) / len(ref_ids)) if ref_ids else 0.0
            citation_precision = (len(overlap) / len(cited_ids)) if cited_ids else 0.0

            # ── Search efficiency ────────────────────────────────────
            search_calls = _count_search_calls(completion)
            if search_calls <= 2:
                efficiency_raw = 1.0
            elif search_calls == 3:
                efficiency_raw = 0.5
            else:
                efficiency_raw = 0.0

            # ── LLM judge: correctness & conciseness ────────────────
            prompt = str(kwargs.get("prompt") or kwargs.get("question") or "")
            gt_str = str(ground_truth or "")

            correctness_raw, conciseness_raw = 0.0, 0.0
            if self._judge_base_url and self._judge_api_key:
                correctness_raw, conciseness_raw = await asyncio.gather(
                    _judge_rubric(
                        question=prompt,
                        ground_truth=gt_str,
                        response=answer_block,
                        rubric_title="Answer correctness",
                        rubric_description="Response correctly answers the question and is factually consistent with the reference answer.",
                        judge_model=self._judge_model,
                        judge_base_url=self._judge_base_url,
                        judge_api_key=self._judge_api_key,
                    ),
                    _judge_rubric(
                        question=prompt,
                        ground_truth=gt_str,
                        response=answer_block,
                        rubric_title="Answer conciseness",
                        rubric_description="Response is concise and avoids unnecessary verbosity while directly answering the question.",
                        judge_model=self._judge_model,
                        judge_base_url=self._judge_base_url,
                        judge_api_key=self._judge_api_key,
                    ),
                )

            # ── Gating: conciseness only counts if answer is correct,
            #    search efficiency only if answer + citations are correct
            correctness_ok = correctness_raw > 0

            return {
                "answer_correctness": W_CORRECTNESS * correctness_raw,
                "conciseness": W_CONCISENESS * conciseness_raw if correctness_ok else 0.0,
                "citation_recall": W_CITATION_RECALL * citation_recall,
                "citation_precision": W_CITATION_PRECISION * citation_precision,
                "search_efficiency": (
                    W_SEARCH_EFFICIENCY * efficiency_raw
                    if (correctness_ok and citation_precision > 0 and citation_recall > 0)
                    else 0.0
                ),
            }

        except Exception:
            return zero

---
## Step 4: Launch Training

Upload datasets + environment and start the RL training job on the CGFT platform.

In [ ]:
from cgft.trainer.pipeline import train

experiment_id = train(
    env_class=MyEmailSearchEnv,
    env_args={
        "api_key": API_KEY,
        "corpus_id": source.corpus_id,
        "base_url": BASE_URL,
        "judge_base_url": JUDGE_BASE_URL,
        "judge_api_key": JUDGE_API_KEY,
        "judge_model": JUDGE_MODEL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix="email-rag",
    api_key=API_KEY,
    base_url=BASE_URL,
    pip_dependencies=["openai", "aiohttp"],
    experiment_name=EXPERIMENT_NAME,
)

print(f"\nExperiment launched! ID: {experiment_id}")
print(f"Monitor at: {BASE_URL}/experiments/{experiment_id}")

---
## What to Expect During Training

**Early training** — Rewards will be noisy for the first few dozen steps. This is normal.

**Healthy trajectory** — Look for `pass@k` climbing first (the model learns to solve more prompts),
then average reward follows (it gets better at each prompt). Both should eventually plateau.

**Warning signs:**
- `pass@k` flat while reward increases → overfitting to a narrow set of easy prompts
- Both stagnate early → training data may be too hard or reward signal too sparse

---
## Customization Guide

### Reward weights
Adjust `W_CORRECTNESS`, `W_CONCISENESS`, `W_CITATION_RECALL`, `W_CITATION_PRECISION`, and
`W_SEARCH_EFFICIENCY` at the top of the environment cell. Higher weight = more influence on training.

### Adding tools
Add more tools to `self._tools` in `__init__`. Each tool is a `(ToolDefinition, async_callable)` pair.
For example, you could add a calendar lookup or contact search tool.

### Changing the reward function
Override `compute_reward` to add your own scoring logic. The return value is a dict of named
reward components — the trainer sums them for the total reward.

### Using Turbopuffer instead of Corpus API
Replace `CorporaChunkSource` with `TpufChunkSource` from `cgft.corpus.turbopuffer.source`
and update the search tool to query Turbopuffer.